# Lab9 Part 2. RAG Tutorial


## 1. Dense Passage Retrieval

In this example, we will use <A HREF="https://sbert.net/">Sentence-BERT</A> embeddings to encode both the query and the documents.

In [ ]:
import os
import numpy as np

from transformers import AutoTokenizer, AutoModel
import torch

# Load model & tokenizer
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# To encode a sentence:
#sentence = "This is an example sentence."
#inputs = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True)
#with torch.no_grad():
#    outputs = model(**inputs)
#    embeddings = outputs.last_hidden_state.mean(dim=1)  # Simple mean pooling
#print(embeddings)

data_directory = 'NFCorpus'

documents = []
# Function to read the text files and encode them
def encode_documents(data_directory, model):
    ids = []
    embeddings = []
    for filename in os.listdir(data_directory):
        if filename.endswith('.txt'):
            file_path = os.path.join(data_directory, filename)
            with open(file_path, 'r', encoding='utf-8') as file:
                lines = file.readlines()
                for line in lines:
                    docid, addr, title, content = line.split('\t')
                    document = f"{title}\n{content}"
                    documents.append(document)
                    ids.append(docid)
                    inputs = tokenizer(document, return_tensors="pt", padding=True, truncation=True)
                    with torch.no_grad():
                        outputs = model(**inputs)
                        embedding = outputs.last_hidden_state.mean(dim=1)
                    embeddings.append(embedding)
    return ids, embeddings

# Encode the documents
ids, embeddings = encode_documents(data_directory, model)

# Save the embeddings and IDs
np.save('embeddings.npy', embeddings)
np.save('ids.npy', np.array(ids))


Now, we'll load the embeddings and use them to perform similarity searches with new queries.

In [ ]:
import numpy as np
import torch.nn.functional as F

# Load the embeddings and IDs
embeddings = np.load('embeddings.npy')
ids = np.load('ids.npy')
embeddings = embeddings.squeeze(1) 

def search(query, model, tokenizer, embeddings, ids, top_k=5, device='cpu', normalize=True):
    """
    Perform dense retrieval over precomputed document embeddings.

    Args:
        query (str): The query string.
        model: Transformer model (e.g., BERT).
        tokenizer: Corresponding tokenizer.
        embeddings (np.ndarray or torch.Tensor): Precomputed document embeddings.
        ids (list): Document identifiers corresponding to embeddings.
        top_k (int): Number of top results to return.
        device (str): 'cuda' or 'cpu'.
        normalize (bool): Whether to normalize vectors before cosine sim.

    Returns:
        List of tuples: (doc_id, similarity_score)
    """

    # Ensure embeddings are torch tensor
    if isinstance(embeddings, np.ndarray):
        embeddings = torch.tensor(embeddings, dtype=torch.float)

    embeddings = embeddings.to(device)

    # Encode query
    model.eval()
    inputs = tokenizer(query, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        query_embedding = outputs.last_hidden_state.mean(dim=1)  # mean pooling

    if normalize:
        query_embedding = F.normalize(query_embedding, p=2, dim=1)
        embeddings = F.normalize(embeddings, p=2, dim=1)

    # Compute cosine similarity
    sim_scores = torch.matmul(query_embedding, embeddings.T).squeeze(0)  # [num_docs]

    # Get top-k indices
    top_k_indices = torch.topk(sim_scores, k=top_k).indices.cpu().numpy()

    # Prepare results
    results = [(ids[i], float(sim_scores[i]), documents[i]) for i in top_k_indices]
    return results

# Example query
query = "milk calcium"
search(query, model, tokenizer, embeddings, ids)

## 2. Using LLMs to explain results

Now we will retrieve and then ask a LLM to validate the result:

The LLM will tell if the result is good and why


First of all, ensure that Ollama is running (type `ollama serve` in the console).

In [ ]:
import ollama

def get_response(prompt, model="llama3.2"):

    response = ollama.chat(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response["message"]["content"]

And we can query the llm like this:

In [ ]:
response=get_response("What is a LLM?")
print(response)

Now we can implement the explanations for the output of our retrieval system

In [ ]:
# Function to perform a search with llm_validation
def search_val(query, model, tokenizer, embeddings, ids, top_k=5, device='cpu', normalize=True):
    """
    Perform dense retrieval over precomputed document embeddings.

    Args:
        query (str): The query string.
        model: Transformer model (e.g., BERT).
        tokenizer: Corresponding tokenizer.
        embeddings (np.ndarray or torch.Tensor): Precomputed document embeddings.
        ids (list): Document identifiers corresponding to embeddings.
        top_k (int): Number of top results to return.
        device (str): 'cuda' or 'cpu'.
        normalize (bool): Whether to normalize vectors before cosine sim.

    Returns:
        List of tuples: (doc_id, similarity_score)
    """

    # Ensure embeddings are torch tensor
    if isinstance(embeddings, np.ndarray):
        embeddings = torch.tensor(embeddings, dtype=torch.float)

    embeddings = embeddings.to(device)

    # Encode query
    model.eval()
    inputs = tokenizer(query, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        query_embedding = outputs.last_hidden_state.mean(dim=1)  # mean pooling

    if normalize:
        query_embedding = F.normalize(query_embedding, p=2, dim=1)
        embeddings = F.normalize(embeddings, p=2, dim=1)

    # Compute cosine similarity
    sim_scores = torch.matmul(query_embedding, embeddings.T).squeeze(0)  # [num_docs]

    # Get top-k indices
    top_k_indices = torch.topk(sim_scores, k=top_k).indices.cpu().numpy()

    # out results
    for i in top_k_indices:
        print("ID:", ids[i])
        print("Score: ", float(sim_scores[i]), documents[i])
        response = get_response('Is the following document: "'+documents[i]+'" pertinent to the query "'+query+'" and why?')
        print("LLM explanation:", response)

In [ ]:
# Example query
query = "milk calcium"
search_val(query, model, tokenizer, embeddings, ids)